In [1]:
import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path
from astropy.io import fits
from astropy.wcs import WCS
import astropy.units as u
import astropy.constants as c
import pyneb as pn
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord, SkyOffsetFrame
from scipy.stats import linregress
from astropy.visualization import AsinhStretch, PercentileInterval
from skimage.segmentation import find_boundaries
from matplotlib import cm
from skimage.measure import find_contours
from matplotlib.patches import ConnectionPatch
from scipy.optimize import curve_fit
from dataclasses import dataclass
from scipy.optimize import brentq
from scipy import odr

In [2]:
# Folder that contains per-field catalogs
CATALOG_DIR = "CATALOGS/flux_catalogs"   # change if needed
PATTERN = "flux_catalog_*.csv"  # matches flux_catalog_{field}.csv

files = sorted(glob.glob(os.path.join(CATALOG_DIR, PATTERN)))
print(f"Found {len(files)} catalog files.")

dfs = []
for fp in files:
    df = pd.read_csv(fp)

    # Ensure there's a 'field' column; if not, parse it from filename
    if "field" not in df.columns:
        base = os.path.basename(fp)
        # expects flux_catalog_{field}.csv
        field = base.replace("flux_catalog_", "").replace(".csv", "")
        df["field"] = field

    dfs.append(df)

# Stack catalogs (adds new rows for each field)
all_catalog = pd.concat(dfs, axis=0, ignore_index=True, sort=False)

print("Combined catalog shape:", all_catalog.shape)

# Save combined catalog
output_path = os.path.join(CATALOG_DIR, "total_flux_catalog.csv")
all_catalog.to_csv(output_path, index=False)

print("Saved combined catalog to:", output_path)

Found 9 catalog files.
Combined catalog shape: (6508, 151)
Saved combined catalog to: CATALOGS/flux_catalogs/total_flux_catalog.csv


# Then, add derived quantities based on the line ratios

# Ionization Parameter based on OIII/OII

In [3]:
C_CM_S = c.c.to_value(u.cm / u.s)  # speed of light in cm/s

def _col(prefix, name, suffix):
    f = f"{prefix}_{name}_{suffix}"
    e = f"{prefix}_{name}_e_{suffix}"
    return f, e

def safe_normal(rng, mu, sigma, n):
    x = rng.normal(mu, sigma, n)
    return np.where(np.isfinite(x), x, np.nan)

def kk04_logq_from_logO32_and_Z(logO32, Z_12logOH):
    """
    KK04 eq. (13) as shown in Kobulnicky & Kewley (2004) PDF view.
    Inputs:
      logO32 = y
      Z_12logOH = z = 12 + log(O/H)
    Returns:
      log10(q [cm/s])
    """
    y = logO32
    z = Z_12logOH
    num = 32.81 - 1.153*(y**2) + z*(-3.396 - 0.025*y + 0.1444*(y**2))
    den = 4.603 - 0.3119*y - 0.163*(y**2) + z*(-0.48 + 0.0271*y + 0.02037*(y**2))
    return num / den

def add_logU_KK04(
    df,
    n_mc=2000,
    seed=123,
    metallicity_cal="M13_O3N2",   # "M13_O3N2" (Marino+2013) or "PP04_O3N2"
    Z_intrinsic_sigma_dex=0.18,   # optional: add intrinsic scatter in metallicity (dex)
    apply_Z_intrinsic_scatter=True
):
    """
    Adds:
      - O32, logO32 (and errs)
      - O3N2, Z_12logOH (and errs)
      - logq_KK04, logU_KK04 and 1-sigma uncertainties from Monte Carlo

    Notes:
      - Uses extinction-corrected (I_) by default.
      - Requires: [OII]3727, [OIII]5007, Hbeta, Halpha, [NII]6583
    """
    out = df.copy()
    prefix = "F"

    oii, oii_e = _col(prefix, "[OII]3727", "sum_dered")
    # o3_4959, o3_4959_e = _col(prefix, "[OIII]4959", "sum_dered")
    o3_5007, o3_5007_e = _col(prefix, "[OIII]5007", "sum_dered")
    hb, hb_e = _col(prefix, "Hbeta", "sum_dered")
    ha, ha_e = _col(prefix, "Halpha", "sum_dered")
    nii, nii_e = _col(prefix, "[NII]6583", "sum_dered")

    rng = np.random.default_rng(seed)

    # Storage
    O32_med = np.full(len(out), np.nan)
    O32_sig = np.full(len(out), np.nan)
    logO32_med = np.full(len(out), np.nan)
    logO32_sig = np.full(len(out), np.nan)

    O3N2_med = np.full(len(out), np.nan)
    O3N2_sig = np.full(len(out), np.nan)
    Z_med = np.full(len(out), np.nan)
    Z_sig = np.full(len(out), np.nan)

    logq_med = np.full(len(out), np.nan)
    logq_sig = np.full(len(out), np.nan)
    logU_med = np.full(len(out), np.nan)
    logU_sig = np.full(len(out), np.nan)

    # Loop row-by-row for robust MC with clipping
    for i in range(len(out)):
        vals = out.loc[i, [oii, oii_e, o3_5007, o3_5007_e, hb, hb_e, ha, ha_e, nii, nii_e]].astype(float).to_numpy()
        if not np.all(np.isfinite(vals)):
            continue

        f_oii, e_oii,f_5007, e_5007, f_hb, e_hb, f_ha, e_ha, f_nii, e_nii = vals
        # require positive fluxes and non-negative errors
        if (f_oii<=0) or (f_5007<=0) or (f_hb<=0) or (f_ha<=0) or (f_nii<=0):
            continue
        if (e_oii<0) or (e_5007<0) or (e_hb<0) or (e_ha<0) or (e_nii<0):
            continue

        # Draw fluxes; clip to small positive to avoid log issues
        d_oii   = np.clip(safe_normal(rng, f_oii,   e_oii,   n_mc), 1e-30, None)
        # d_4959  = np.clip(safe_normal(rng, f_4959,  e_4959,  n_mc), 1e-30, None)
        d_5007  = np.clip(safe_normal(rng, f_5007,  e_5007,  n_mc), 1e-30, None)
        d_hb    = np.clip(safe_normal(rng, f_hb,    e_hb,    n_mc), 1e-30, None)
        d_ha    = np.clip(safe_normal(rng, f_ha,    e_ha,    n_mc), 1e-30, None)
        d_nii   = np.clip(safe_normal(rng, f_nii,   e_nii,   n_mc), 1e-30, None)

        # O32
        
        #assume constant OIII ratio of 2.98 to get total OIII from 5007 alone (since 4959 is often weak)
        d_o32 = ((1+1/2.98)*d_5007) / d_oii
        m1 = np.isfinite(d_o32) & (d_o32 > 0)
        if m1.sum() < 50:
            continue
        d_o32 = d_o32[m1]
        d_logO32 = np.log10(d_o32)

        # O3N2 = log10( (OIII5007/Hb) * (Ha/NII6583) )
        d_o3n2 = np.log10((d_5007/d_hb) * (d_ha/d_nii))
        m2 = np.isfinite(d_o3n2)
        if m2.sum() < 50:
            continue
        d_o3n2 = d_o3n2[m2]

        # Metallicity from O3N2
        if metallicity_cal == "M13_O3N2":
            # Marino+2013: 12+log(O/H) = 8.533 - 0.214*O3N2
            d_Z = 8.533 - 0.214 * d_o3n2
        elif metallicity_cal == "PP04_O3N2":
            # Pettini & Pagel 2004: 12+log(O/H) = 8.73 - 0.32*O3N2  (valid in their stated range)
            d_Z = 8.73 - 0.32 * d_o3n2
        else:
            raise ValueError("metallicity_cal must be 'M13_O3N2' or 'PP04_O3N2'.")

        # optionally fold in intrinsic metallicity-calibration scatter (dex) as extra random term
        if apply_Z_intrinsic_scatter and (Z_intrinsic_sigma_dex is not None) and (Z_intrinsic_sigma_dex > 0):
            d_Z = d_Z + rng.normal(0.0, Z_intrinsic_sigma_dex, size=d_Z.size)

        # Now compute logq (KK04) and logU = logq - log10(c)
        # Need to align sizes: use common subset length by trimming to min
        n = min(d_logO32.size, d_Z.size)
        y = d_logO32[:n]
        z = d_Z[:n]
        d_logq = kk04_logq_from_logO32_and_Z(y, z)
        d_logU = d_logq - np.log10(C_CM_S)

        m3 = np.isfinite(d_logq) & np.isfinite(d_logU)
        if m3.sum() < 50:
            continue
        d_logq = d_logq[m3]
        d_logU = d_logU[m3]

        # Summaries: median and 16–84 half-width as robust 1-sigma
        def med_sig(x):
            p16, p50, p84 = np.nanpercentile(x, [16, 50, 84])
            return p50, 0.5*(p84 - p16)

        O32_med[i], O32_sig[i] = med_sig(d_o32)
        logO32_med[i], logO32_sig[i] = med_sig(d_logO32)

        O3N2_med[i], O3N2_sig[i] = med_sig(d_o3n2)
        Z_med[i], Z_sig[i] = med_sig(d_Z[:n])

        logq_med[i], logq_sig[i] = med_sig(d_logq)
        logU_med[i], logU_sig[i] = med_sig(d_logU)

    # Write columns
    out["O32"] = O32_med
    out["O32_e"] = O32_sig
    out["logO32"] = logO32_med
    out["logO32_e"] = logO32_sig

    out["O3N2"] = O3N2_med
    out["O3N2_e"] = O3N2_sig
    out["Z_12logOH"] = Z_med
    out["Z_12logOH_e"] = Z_sig

    out["logq_KK04"] = logq_med
    out["logq_KK04_e"] = logq_sig
    out["logU_KK04"] = logU_med
    out["logU_KK04_e"] = logU_sig

    out["logU_flag"] = np.where(np.isfinite(out["logU_KK04"]), "ok", "invalid")

    out["logU_meta_cal"] = metallicity_cal
    out["logU_meta_Zscatter_dex"] = (Z_intrinsic_sigma_dex if apply_Z_intrinsic_scatter else 0.0)

    return out

# --- Usage ---
cat = add_logU_KK04(all_catalog.copy(), n_mc=2000, seed=123, metallicity_cal="M13_O3N2")


In [4]:
# save the catalog with the derived quantities
derived_output_path = os.path.join(CATALOG_DIR, "total_flux_catalog_with_derived.csv")
cat.to_csv(derived_output_path, index=False)
print('Number of columns:', len(cat.columns))
print("Saved combined catalog with ionization parameter:", derived_output_path)

Number of columns: 166
Saved combined catalog with ionization parameter: CATALOGS/flux_catalogs/total_flux_catalog_with_derived.csv


# $n_e$ using [SII] ratio

In [5]:
TE_DEFAULT = 1.0e4  # K, typical assumption for H II regions
use_dereddened = True  # prefer I_ columns if they exist

# Column names in your catalog
c_6716_I, c_6731_I = "F_[SII]6716_sum_dered", "F_[SII]6731_sum_dered"
c_6716_F, c_6731_F = "F_[SII]6716_sum", "F_[SII]6731_sum"

c_6716_Ie, c_6731_Ie = "F_[SII]6716_e_sum_dered", "F_[SII]6731_e_sum_dered"
c_6716_Fe, c_6731_Fe = "F_[SII]6716_e_sum", "F_[SII]6731_e_sum"

df = cat.copy()

# -----------------------------
# Choose which flux columns to use
# -----------------------------
if use_dereddened and (c_6716_I in df.columns) and (c_6731_I in df.columns):
    c6716, c6731 = c_6716_I, c_6731_I
    c6716e, c6731e = (c_6716_Ie if c_6716_Ie in df.columns else None), (c_6731_Ie if c_6731_Ie in df.columns else None)
else:
    c6716, c6731 = c_6716_F, c_6731_F
    c6716e, c6731e = (c_6716_Fe if c_6716_Fe in df.columns else None), (c_6731_Fe if c_6731_Fe in df.columns else None)

# -----------------------------
# Compute the SII ratio
# -----------------------------


df["SII_ratio_6716_6731"] = df[c6716] / df[c6731]

# Optional: ratio uncertainty from standard propagation
if (c6716e is not None) and (c6731e is not None):
    ratio = df["SII_ratio_6716_6731"].to_numpy(dtype=float)
    f1 = df[c6716].to_numpy(dtype=float)
    f2 = df[c6731].to_numpy(dtype=float)
    e1 = df[c6716e].to_numpy(dtype=float)
    e2 = df[c6731e].to_numpy(dtype=float)

    # sigma(r) / r = sqrt( (e1/f1)^2 + (e2/f2)^2 )
    with np.errstate(divide="ignore", invalid="ignore"):
        df["SII_ratio_6716_6731_e"] = np.abs(ratio) * np.sqrt((e1 / f1) ** 2 + (e2 / f2) ** 2)

In [6]:
# df.columns

In [7]:
# -----------------------------
# PyNeb: compute electron density from the ratio
# -----------------------------
S2 = pn.Atom("S", 2)  # S II

# Guard against invalid ratios / non-positive fluxes
valid = np.isfinite(df["SII_ratio_6716_6731"]) & (df[c6716] > 0) & (df[c6731] > 0)

ne = np.full(len(df), np.nan, dtype=float)
ne[valid.to_numpy()] = S2.getTemDen(
    df.loc[valid, "SII_ratio_6716_6731"].to_numpy(dtype=float),
    tem=TE_DEFAULT,
    wave1=6716,
    wave2=6731
)

df["ne_SII_cm3"] = ne  # electron density in cm^-3

In [8]:
# -----------------------------
# Optional: Monte Carlo uncertainty on ne (uses ratio uncertainty if available)
# -----------------------------
def monte_carlo_ne_from_ratio(r, rerr, te=TE_DEFAULT, nmc=200, seed=0):
    """
    Returns median ne and +/-1sigma (16th/84th percentiles) from MC draws.
    """
    rng = np.random.default_rng(seed)
    draws = rng.normal(loc=r, scale=rerr, size=nmc)

    # Remove non-physical draws
    draws = draws[np.isfinite(draws) & (draws > 0)]
    if draws.size == 0:
        return np.nan, np.nan, np.nan

    ne_draws = S2.getTemDen(draws, tem=te, wave1=6716, wave2=6731)
    ne_draws = np.atleast_1d(ne_draws)

    # print(f"MC ratio draws: {draws}")
    # print(f"MC ne draws: {ne_draws}")

    ne_draws = ne_draws[np.isfinite(ne_draws) & (ne_draws > 0)]
    if ne_draws.size == 0:
        return np.nan, np.nan, np.nan

    p16, p50, p84 = np.percentile(ne_draws, [16, 50, 84])
    return p50, (p50 - p16), (p84 - p50)

if "SII_ratio_6716_6731_e" in df.columns:
    out = np.array(
        [monte_carlo_ne_from_ratio(r, e, te=TE_DEFAULT, nmc=3, seed=i)
         if np.isfinite(r) and np.isfinite(e) and (e > 0)
         else (np.nan, np.nan, np.nan)
         for i, (r, e) in enumerate(zip(df["SII_ratio_6716_6731"], df["SII_ratio_6716_6731_e"]))],
        dtype=float
    )
    df["ne_SII_cm3_mc"] = out[:, 0]
    df["ne_SII_cm3_mc_minus"] = out[:, 1]
    df["ne_SII_cm3_mc_plus"] = out[:, 2]



In [9]:
# save the catalog with the derived quantities
derived_output_path = os.path.join(CATALOG_DIR, "total_flux_catalog_with_derived.csv")
df.to_csv(derived_output_path, index=False)
print('Number of columns:', len(df.columns))
print("Saved combined catalog with electron densities:", derived_output_path)

Number of columns: 172
Saved combined catalog with electron densities: CATALOGS/flux_catalogs/total_flux_catalog_with_derived.csv


# Many different metallicity indicators

In [10]:
# Classify symmtry of regions based on difference between R16, R50, R84
def classify_symmetry(row, threshold=0.2):
    
    R16 = row['radius_p16_pc']
    R50 = row['radius_p50_pc']
    R84 = row['radius_p84_pc']

    if not np.isfinite(R16) or not np.isfinite(R50) or not np.isfinite(R84):
        return 'unknown'

    # Compute asymmetry metric
    if R84 != R16:
        asymmetry = abs(R84 - R16) / (R84 + R16)
    else:
        asymmetry = 0.0  # perfectly symmetric if R84 == R16

    if asymmetry < threshold:
        return 'symmetric'
    else:
        return 'asymmetric'
    
df['symmetry_class'] = df.apply(classify_symmetry, axis=1)

#print how many regions are classified as symmetric, asymmetric, and unknown
symmetry_counts = df['symmetry_class'].value_counts()
print("Symmetry classification counts:")
print(symmetry_counts)

#save the catalog with the symmetry classification
derived_output_path = os.path.join(CATALOG_DIR, "total_flux_catalog_with_derived_and_symmetry.csv")
df.to_csv(derived_output_path, index=False)
print('Number of columns:', len(df.columns))
print("Saved combined catalog with symmetry classification:", derived_output_path)

Symmetry classification counts:
symmetry_class
asymmetric    5381
symmetric     1127
Name: count, dtype: int64
Number of columns: 173
Saved combined catalog with symmetry classification: CATALOGS/flux_catalogs/total_flux_catalog_with_derived_and_symmetry.csv


In [11]:
#Brazzini+2024 metallicity calibrations

def poly_eval(coeffs, z):
    # coeffs = [c0, c1, ..., cn]
    return np.polyval(coeffs[::-1], z)

def invert_logR_to_Z(coeffs, logR_obs, z_min=6.5, z_max=9.5, ngrid=2000):
    """
    Robust inversion of logR = poly(Z) for Z in [z_min, z_max].
    Returns np.nan if logR_obs is not achievable in this Z-range.
    """
    if not np.isfinite(logR_obs):
        return np.nan

    zgrid = np.linspace(z_min, z_max, ngrid)
    fgrid = poly_eval(coeffs, zgrid) - logR_obs

    # If all fgrid are positive or all negative, there is no crossing -> no solution in range.
    if np.all(~np.isfinite(fgrid)):
        return np.nan

    finite = np.isfinite(fgrid)
    zgrid = zgrid[finite]
    fgrid = fgrid[finite]
    if zgrid.size < 2:
        return np.nan

    # Look for sign changes between adjacent grid points
    s = np.sign(fgrid)
    # sign change where s[i]*s[i+1] <= 0 and neither is 0
    idx = np.where(s[:-1] * s[1:] < 0)[0]

    if idx.size == 0:
        # No bracket found. Return the Z with minimum absolute residual as a fallback
        j = np.argmin(np.abs(fgrid))
        return zgrid[j]

    # If multiple brackets, pick the one with the smallest residual near crossing
    # Choose bracket whose mid-point gives smallest |f|
    best_i = None
    best_val = np.inf
    for i in idx:
        midz = 0.5 * (zgrid[i] + zgrid[i+1])
        val = abs(poly_eval(coeffs, midz) - logR_obs)
        if val < best_val:
            best_val = val
            best_i = i

    a, b = zgrid[best_i], zgrid[best_i + 1]

    # Brent root in that bracket
    try:
        return brentq(lambda z: poly_eval(coeffs, z) - logR_obs, a, b, maxiter=200)
    except ValueError:
        # fallback
        j = np.argmin(np.abs(fgrid))
        return zgrid[j]

def odr_refine_z(coeffs, logR_obs, z0, sx=1.0, sy=1.0):
    if not (np.isfinite(z0) and np.isfinite(logR_obs)):
        return np.nan

    def f(beta, x):
        z = beta[0]
        return poly_eval(coeffs, z) + 0.0*x

    model = odr.Model(f)
    x = np.array([0.0])
    y = np.array([logR_obs])
    data = odr.RealData(x, y, sx=np.array([sx]), sy=np.array([sy]))
    out = odr.ODR(data, model, beta0=[z0]).run()
    return out.beta[0]

@dataclass(frozen=True)
class Calibration:
    name: str
    coeffs: np.ndarray
    R_func: callable

def build_calibrations():
    return [
        Calibration(
            name="N2S2Halpha_Brazzini2024",
            coeffs=np.array([0.24, 2.21, 0.76]),
            R_func=lambda N2, S2, R3, R2, R23: N2 / S2 * (N2 ** 0.264),
        ),
        Calibration(
            name="N2_Brazzini2024",
            coeffs=np.array([-0.41, 0.57, -4.91, -5.81, -1.95]),
            R_func=lambda N2, S2, R3, R2, R23: N2,
        ),
        Calibration(
            name="O3N2_Brazzini2024",
            coeffs=np.array([-0.51, -7.74, -6.12, -1.60]),
            R_func=lambda N2, S2, R3, R2, R23: R3 / N2,
        ),
        Calibration(
            name="R3_Brazzini2024",
            coeffs=np.array([-0.84, -5.86, -6.27, -1.95]),
            R_func=lambda N2, S2, R3, R2, R23: R3,
        ),
        
        Calibration(
            name = "R23_Maiolino2008",
            coeffs = np.array([0.7462, -0.7149, -0.9401, -0.6154, -0.2524]),
            R_func = lambda N2, S2, R3, R2, R23: R23
        ),
        
        Calibration(
            name = "N2_Maiolino2008",
            coeffs = np.array([-0.7732, 1.2357, -0.2811, -0.7201, -0.3330]),
            R_func = lambda N2, S2, R3, R2, R23: N2
        ),
        
        #table 2 of Curti+2017
        Calibration(
            name='R3_Curti2017',
            coeffs = np.array([-0.277, -3.549, -3.593, -0.981]),
            R_func = lambda N2, S2, R3, R2, R23: R3
        ),
        
        Calibration(
            name='R23_Curti2017',
            coeffs=np.array([0.527, -1.569, -1.652, -0.421]),
            R_func=lambda N2, S2, R3, R2, R23: R23
        ),
        
        Calibration(
            name='N2_Curti2017',
            coeffs=np.array([-0.489, 1.513, -2.554, -5.293, -2.867]),
            R_func=lambda N2, S2, R3, R2, R23: N2
        ),
        
        Calibration(
            name='O3N2_Curti2017',
            coeffs=np.array([-0.281, -4.765, -2.268]),
            R_func=lambda N2, S2, R3, R2, R23: R3 / N2
        )
    ]

def compute_metallicities(full_catalog, z_min=6.5, z_max=9.5, use_odr=True, sx=1.0, sy=1.0):
    Ha = np.asarray(full_catalog["F_Halpha_sum_dered"], float)
    Hb = np.asarray(full_catalog["F_Hbeta_sum_dered"], float)
    NII = np.asarray(full_catalog["F_[NII]6583_sum_dered"], float)
    SII1 = np.asarray(full_catalog["F_[SII]6716_sum_dered"], float)
    SII2 = np.asarray(full_catalog["F_[SII]6731_sum_dered"], float)
    OIII = np.asarray(full_catalog["F_[OIII]5007_sum_dered"], float)
    OII = np.asarray(full_catalog["F_[OII]3727_sum_dered"], float)

    with np.errstate(divide="ignore", invalid="ignore"):
        N2 = NII / Ha
        S2 = (SII1 + SII2) / Ha
        R3 = OIII / Hb
        R2 = OII / Hb
        R23 = R2 + R3

    calibrations = build_calibrations()
    out = {}

    for cal in calibrations:
        with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
            R = cal.R_func(N2, S2, R3, R2, R23)

        # Only compute logR where R>0
        logR = np.full_like(R, np.nan, dtype=float)
        goodR = np.isfinite(R) & (R > 0)
        logR[goodR] = np.log10(R[goodR])

        Z = np.full_like(logR, np.nan, dtype=float)
        for i, logR_i in enumerate(logR):
            if not np.isfinite(logR_i):
                continue

            z0 = invert_logR_to_Z(cal.coeffs, logR_i, z_min=z_min, z_max=z_max)
            if use_odr and np.isfinite(z0):
                z1 = odr_refine_z(cal.coeffs, logR_i, z0, sx=sx, sy=sy)
                Z[i] = z1 if np.isfinite(z1) else z0
            else:
                Z[i] = z0

        out[cal.name] = Z

        # quick summary per method
        finite_frac = np.isfinite(Z).mean()
        print(f"{cal.name}: finite Z fraction = {finite_frac:.3f}")

    return out


# #perform a snr cut of 3 on all lines used in the calibrations
# snr_cols = ['SNR_Halpha', 'SNR_Hbeta', 'SNR_[NII]6583', 'SNR_[SII]6716', 'SNR_[SII]6731', 'SNR_[OIII]5007', 'SNR_[OII]3727']
# snr_cut = 3.0
# mask = np.ones(len(catalog), dtype=bool)
# for col in snr_cols:
#     mask &= catalog[col] >= snr_cut


# print(f"Computing metallicities for {np.sum(mask)} sources with SNR >= {snr_cut} in all lines.")

Z_dict = compute_metallicities(df, use_odr=True)
Z_N2  = Z_dict["N2_Brazzini2024"] + 8.69
Z_O3N2 = Z_dict["O3N2_Brazzini2024"]+ 8.69
Z_N2S2Halpha = Z_dict["N2S2Halpha_Brazzini2024"] + 8.69
Z_R3 = Z_dict["R3_Brazzini2024"] + 8.69

Z_R23_Maiolino2008 = Z_dict["R23_Maiolino2008"] + 8.69
Z_N2_Maiolino2008 = Z_dict["N2_Maiolino2008"] + 8.69

Z_R23_Curti2017 = Z_dict["R23_Curti2017"] + 8.69
Z_R3_Curti2017 = Z_dict["R3_Curti2017"] + 8.69
Z_N2_Curti2017 = Z_dict["N2_Curti2017"] + 8.69
Z_O3N2_Curti2017 = Z_dict["O3N2_Curti2017"] + 8.69

#add them to the catalog
df["Z_N2_Brazzini2024"] = Z_N2
df["Z_O3N2_Brazzini2024"] = Z_O3N2
df["Z_N2S2Halpha_Brazzini2024"] = Z_N2S2Halpha
df["Z_R3_Brazzini2024"] = Z_R3 
df["Z_R23_Maiolino2008"] = Z_R23_Maiolino2008
df["Z_N2_Maiolino2008"] = Z_N2_Maiolino2008
df["Z_R23_Curti2017"] = Z_R23_Curti2017
df["Z_R3_Curti2017"] = Z_R3_Curti2017
df["Z_N2_Curti2017"] = Z_N2_Curti2017
df["Z_O3N2_Curti2017"] = Z_O3N2_Curti2017

N2S2Halpha_Brazzini2024: finite Z fraction = 0.949
N2_Brazzini2024: finite Z fraction = 0.959
O3N2_Brazzini2024: finite Z fraction = 0.936
R3_Brazzini2024: finite Z fraction = 0.939
R23_Maiolino2008: finite Z fraction = 0.951
N2_Maiolino2008: finite Z fraction = 0.959
R3_Curti2017: finite Z fraction = 0.939
R23_Curti2017: finite Z fraction = 0.951
N2_Curti2017: finite Z fraction = 0.959
O3N2_Curti2017: finite Z fraction = 0.936


In [14]:
# Callibrations from Pilyugin & Grebel (2016)


mask = np.ones(len(df), dtype=bool)
Ha = np.asarray(df[mask]["F_Halpha_sum_dered"], float)
Hb = np.asarray(df[mask]["F_Hbeta_sum_dered"], float)
NII = np.asarray(df[mask]["F_[NII]6583_sum_dered"], float)
SII1 = np.asarray(df[mask]["F_[SII]6716_sum_dered"], float)
SII2 = np.asarray(df[mask]["F_[SII]6731_sum_dered"], float)
SII = SII1 + SII2
OIII = np.asarray(df[mask]["F_[OIII]5007_sum_dered"], float)
OII = np.asarray(df[mask]["F_[OII]3727_sum_dered"], float)

N2 = NII / Ha
S2 = (SII1 + SII2) / Ha
R3 = (1+1/2.89) * OIII / Hb #correct for 4959 line??
# R3 = OIII / Hb
R2 = OII / Hb
        
#R calibration

#logN2>-0.6
O_H_R_Pilyugin2016_highN2 = 8.589 + 0.022 * np.log10(R3 / R2) + 0.399 * np.log10(N2) + (-0.137 + 0.164 * np.log10(R3 / R2) + 0.589 * np.log10(N2)) * np.log10(R2)
#logN2<=-0.6
O_H_R_Pilyugin2016_lowN2 = 7.932 + 0.944 * np.log10(R3 / R2) + 0.695 * np.log10(N2) + (0.970 - 0.291 * np.log10(R3 / R2) - 0.019 * np.log10(N2)) * np.log10(R2)

#S_calibration

#logN2>-0.6
O_H_S_Pilyugin2016_highN2 = 8.424 + 0.030 * np.log10(R3 / S2) + 0.751 * np.log10(N2) + (-0.349 + 0.182 * np.log10(R3 / S2) + 0.508 * np.log10(N2)) * np.log10(S2)

#logN2<=-0.6
O_H_S_Pilyugin2016_lowN2 = 8.072 + 0.789 * np.log10(R3 / S2) + 0.726 * np.log10(N2) + (1.069 - 0.170 * np.log10(R3 / S2) + 0.022 * np.log10(N2)) * np.log10(S2)

#R23 from Kobulnicky & Kewley (2004) equation 18
R23 = (OIII + OII) / Hb
O32 = OIII / OII
x = np.log10(R23)
y = np.log10(O32)
O_H_KK2004 = 9.11 - 0.218 * x - 0.0587 * x**2 - 0.330 * x**3 -0.199*x**4 - y * (0.00235 - 0.1105 * x - 0.051 * x**2 - 0.04085 * x**3 - 0.003585 * x**4)  


#Calibrations from Kewley & Dopita (2002)

#equation 7
O_H_NII_KD2002 = np.log10(1.54020 + 1.26602 * NII/OII + 0.167977 * (NII/OII)**2) + 8.93



#Calibrations from Dopita et al. (2016)
y = np.log10(NII/SII) + 0.264 * np.log10(NII/Ha)
O_H_D2016 = 8.77 + y + 0.45 * (y + 0.3)**5
O_H_D2016 = 8.77 + y




#Calibrations from Marino et al. (2013)
#equation 2 - O3N2
O_H_O3N2_M2013 = 8.533 - 0.214 * np.log10(R3 / N2)
#equation 4 - N2
O_H_N2_M2013 = 8.743 + 0.462 * np.log10(N2)


#Chralot & Longhetti (2001) Te-based metallicities (equation 11 from Kewley & Dopita 2002)
O_H_C2001 = np.log10(5.09e-4*(OII/OIII)**0.17 * (NII/SII/0.85)**1.17) + 12.0


#Calibrations from Pettini & Pagel (2004)
#equation 2 N2
O_H_N2_PP2004 = 9.37 + 2.03 * np.log10(N2) + 1.26 * (np.log10(N2))**2 + 0.32 * (np.log10(N2))**3

#equation 3 O3N2
O_H_O3N2_PP2004 = 8.73 - 0.32 * np.log10(R3 / N2)


#Kuzio de Naray et al. (2004) based on McGaugh 1991
x = np.log10(R23)
y = np.log10(O32)

# O_H_N2004 = 



# Tremonti et al. 2004
O_H_T2004 = 9.185 - 0.313 * x - 0.264 * x**2 - 0.321 * x**3


#Brown et al. (2016)
#this was cited in , who just take SSFR=0
#equation 7
O_H_N2_Brown2016 = 9.12 + 0.58 * np.log10(N2)
#equation 8
O_H_O3N2_Brown2016 = 8.98 - 0.32 * np.log10(R3 / N2)
#equation 9
O_H_N2O2_Brown2016 = 9.20 + 0.54 * np.log10(NII / OII)

#N202 from Kewley & Dopita 2002
O_H_N2O2_KD2002 = np.log10(1.54020 + 1.26602 * NII/OII + 0.167977 * (NII/OII)**2) + 8.93



#add these to the catalog
df["Z_R_Pilyugin2016_highN2"] = O_H_R_Pilyugin2016_highN2
df["Z_R_Pilyugin2016_lowN2"] = O_H_R_Pilyugin2016_lowN2
df["Z_S_Pilyugin2016_highN2"] = O_H_S_Pilyugin2016_highN2
df["Z_S_Pilyugin2016_lowN2"] = O_H_S_Pilyugin2016_lowN2
df["Z_R23_KK2004"] = O_H_KK2004
df["Z_NII_KD2002"] = O_H_NII_KD2002
df["Z_D2016"] = O_H_D2016
df["Z_O3N2_M2013"] = O_H_O3N2_M2013
df["Z_N2_M2013"] = O_H_N2_M2013
df["Z_C2001"] = O_H_C2001
df["Z_N2_PP2004"] = O_H_N2_PP2004
df["Z_O3N2_PP2004"] = O_H_O3N2_PP2004
df["Z_N2_Brown2016"] = O_H_N2_Brown2016
df["Z_O3N2_Brown2016"] = O_H_O3N2_Brown2016
df["Z_N2O2_Brown2016"] = O_H_N2O2_Brown2016 
df["Z_N2O2_KD2002"] = O_H_N2O2_KD2002

/var/folders/90/nb83zn5j00bc53x6bcv0jcwm0000gn/T/ipykernel_49816/2962539983.py:23: RuntimeWarning: divide by zero encountered in divide
  O_H_R_Pilyugin2016_highN2 = 8.589 + 0.022 * np.log10(R3 / R2) + 0.399 * np.log10(N2) + (-0.137 + 0.164 * np.log10(R3 / R2) + 0.589 * np.log10(N2)) * np.log10(R2)
/var/folders/90/nb83zn5j00bc53x6bcv0jcwm0000gn/T/ipykernel_49816/2962539983.py:23: RuntimeWarning: invalid value encountered in log10
  O_H_R_Pilyugin2016_highN2 = 8.589 + 0.022 * np.log10(R3 / R2) + 0.399 * np.log10(N2) + (-0.137 + 0.164 * np.log10(R3 / R2) + 0.589 * np.log10(N2)) * np.log10(R2)
/var/folders/90/nb83zn5j00bc53x6bcv0jcwm0000gn/T/ipykernel_49816/2962539983.py:23: RuntimeWarning: divide by zero encountered in log10
  O_H_R_Pilyugin2016_highN2 = 8.589 + 0.022 * np.log10(R3 / R2) + 0.399 * np.log10(N2) + (-0.137 + 0.164 * np.log10(R3 / R2) + 0.589 * np.log10(N2)) * np.log10(R2)
/var/folders/90/nb83zn5j00bc53x6bcv0jcwm0000gn/T/ipykernel_49816/2962539983.py:23: RuntimeWarning: inva

In [15]:
#save the catalog with the derived metallicities
derived_output_path = os.path.join(CATALOG_DIR, "total_flux_catalog_with_derived_and_metallicities.csv")
df.to_csv(derived_output_path, index=False)
print('Number of columns:', len(df.columns))
print("Saved combined catalog with metallicities:", derived_output_path)

Number of columns: 199
Saved combined catalog with metallicities: CATALOGS/flux_catalogs/total_flux_catalog_with_derived_and_metallicities.csv


In [21]:
import os
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.spatial.distance import pdist, squareform

# ============================================================
# USER INPUTS
# ============================================================
# DataFrame already loaded as df
# df = pd.read_csv(...)

x_col = "x"   # pixel x of HII-region centre
y_col = "y"   # pixel y of HII-region centre

# Galaxy centre in the same pixel coordinate system as x_col/y_col
# Replace these with the M33 centre in your SITELLE mosaic/frame.
x0 = 1024.0
y0 = 1024.0

# Output directory
# CATALOG_DIR = "..."

# ============================================================
# M33 + SITELLE GEOMETRY
# ============================================================
pixel_scale_arcsec = 0.32         # SITELLE nominal pixel scale
distance_m33_pc = 883_000.0       # adopt 883 kpc
inclination_deg = 56.0            # commonly adopted for M33
pa_deg = 23.0                     # commonly adopted PA of major axis

# Small-angle conversion
pc_per_arcsec = distance_m33_pc / 206265.0
pc_per_pixel = pixel_scale_arcsec * pc_per_arcsec

print(f"pc/pixel = {pc_per_pixel:.4f}")

# ============================================================
# DEPROJECTION
# ============================================================
def deproject_pixels_to_disk_pc(x, y, x0, y0, pa_deg=23.0, inc_deg=56.0, pc_per_pixel=1.0):
    """
    Convert observed image-plane pixel coordinates to deprojected disk-plane coordinates in pc.

    Convention:
    - x, y: image pixel coordinates
    - x0, y0: galaxy centre in pixels
    - pa_deg: position angle of the major axis (degrees)
    - inc_deg: inclination (degrees), 0 = face-on

    Returns
    -------
    X_pc, Y_pc : deprojected disk-plane coordinates in pc
    """
    dx = np.asarray(x, dtype=float) - x0
    dy = np.asarray(y, dtype=float) - y0

    # Rotate so X' is along major axis.
    # This sign convention is consistent as long as you use the same transform for all points.
    theta = np.deg2rad(pa_deg)
    x_major =  dx * np.cos(theta) + dy * np.sin(theta)
    y_minor = -dx * np.sin(theta) + dy * np.cos(theta)

    # Undo inclination foreshortening along minor axis
    inc = np.deg2rad(inc_deg)
    y_minor_deproj = y_minor / np.cos(inc)

    # Convert to pc
    X_pc = x_major * pc_per_pixel
    Y_pc = y_minor_deproj * pc_per_pixel

    return X_pc, Y_pc

# Compute deprojected coordinates
df["x_deproj_pc"], df["y_deproj_pc"] = deproject_pixels_to_disk_pc(
    df[x_col].values,
    df[y_col].values,
    x0=x0,
    y0=y0,
    pa_deg=pa_deg,
    inc_deg=inclination_deg,
    pc_per_pixel=pc_per_pixel
)

coords_pc = df[["x_deproj_pc", "y_deproj_pc"]].to_numpy()
n_regions = len(coords_pc)

# ============================================================
# LOCAL ENVIRONMENT METRICS
# ============================================================
tree = cKDTree(coords_pc)

# k=6 gives:
#   0th entry = self
#   1st = nearest other neighbour
#   5th = 5th closest other neighbour
k_needed = min(6, n_regions)
distances, indices = tree.query(coords_pc, k=k_needed)

if k_needed == 1:
    distances = distances[:, np.newaxis]

nearest_neighbor_pc = np.full(n_regions, np.nan)
fifth_neighbor_pc = np.full(n_regions, np.nan)

if n_regions >= 2:
    nearest_neighbor_pc = distances[:, 1]

if n_regions >= 6:
    fifth_neighbor_pc = distances[:, 5]

radius_pc = 100.0
n_within_100pc = np.array([
    len(tree.query_ball_point(point, r=radius_pc)) - 1
    for point in coords_pc
], dtype=int)

df["nearest_neighbor_pc_deproj"] = nearest_neighbor_pc
df["n_regions_within_100pc_deproj"] = n_within_100pc
df["distance_5th_closest_pc_deproj"] = fifth_neighbor_pc

# Local projected surface density from 5th neighbour
df["sigma5_per_pc2_deproj"] = np.where(
    np.isfinite(df["distance_5th_closest_pc_deproj"]),
    5.0 / (np.pi * df["distance_5th_closest_pc_deproj"]**2),
    np.nan
)

# ============================================================
# GLOBAL CLUSTERING STATISTICS
# ============================================================

# ---------- 1) Clark-Evans nearest-neighbour ratio ----------
# R < 1 => clustered
# R ~ 1 => random (CSR)
# R > 1 => regular/inhibited
def clark_evans_R(coords):
    n = len(coords)
    if n < 2:
        return np.nan, np.nan, np.nan

    tree = cKDTree(coords)
    d, _ = tree.query(coords, k=2)
    r_obs = np.mean(d[:, 1])

    # area of bounding rectangle in deprojected coordinates
    xmin, ymin = coords.min(axis=0)
    xmax, ymax = coords.max(axis=0)
    area = (xmax - xmin) * (ymax - ymin)

    lam = n / area  # intensity
    r_exp = 1.0 / (2.0 * np.sqrt(lam))
    R = r_obs / r_exp
    return R, r_obs, r_exp

R_ce, nn_obs_mean, nn_exp_mean = clark_evans_R(coords_pc)

# ---------- 2) Ripley's K and Besag's L ----------
# Simple rectangular-window version.
# For publication-quality work, use a footprint/mask and edge correction.
def ripley_K_rect(coords, radii_pc):
    n = len(coords)
    if n < 2:
        return np.full_like(radii_pc, np.nan, dtype=float)

    xmin, ymin = coords.min(axis=0)
    xmax, ymax = coords.max(axis=0)
    area = (xmax - xmin) * (ymax - ymin)

    D = squareform(pdist(coords))
    K = np.zeros_like(radii_pc, dtype=float)

    for i, r in enumerate(radii_pc):
        # count unordered pairs with d <= r, excluding diagonal
        count = np.sum((D <= r) & (D > 0))
        K[i] = area * count / (n * (n - 1))
    return K

radii_pc = np.arange(25, 1001, 25)  # 25 pc to 1000 pc
K_r = ripley_K_rect(coords_pc, radii_pc)
L_r = np.sqrt(K_r / np.pi) - radii_pc  # Besag's L(r) - r form

# ---------- 3) Pair-correlation / shell counts ----------
# g(r) > 1 means excess pairs at scale r
def pair_correlation_rect(coords, r_edges):
    n = len(coords)
    if n < 2:
        return np.full(len(r_edges)-1, np.nan), 0.0

    xmin, ymin = coords.min(axis=0)
    xmax, ymax = coords.max(axis=0)
    area = (xmax - xmin) * (ymax - ymin)
    lam = n / area

    d = pdist(coords)
    counts, _ = np.histogram(d, bins=r_edges)

    shell_areas = np.pi * (r_edges[1:]**2 - r_edges[:-1]**2)

    # expected unordered pair counts in CSR:
    # N_exp ~ 0.5 * n * lam * shell_area
    expected = 0.5 * n * lam * shell_areas
    g_r = counts / expected
    return g_r, lam

r_edges = np.arange(0, 1001, 25)
g_r, lam = pair_correlation_rect(coords_pc, r_edges)
r_centers = 0.5 * (r_edges[:-1] + r_edges[1:])

# ---------- 4) Minimum spanning tree ----------
# Shorter mean MST edge length => more clustered
def mst_mean_edge(coords):
    n = len(coords)
    if n < 2:
        return np.nan
    D = squareform(pdist(coords))
    mst = minimum_spanning_tree(D)
    edges = mst.data
    return np.mean(edges)

mst_mean_pc = mst_mean_edge(coords_pc)

# ---------- 5) Correlation dimension D2 ----------
# Uses cumulative pair counts C(r) ~ r^D2 on a chosen fitting range
def correlation_dimension(coords, r_min=50.0, r_max=500.0, n_bins=20):
    n = len(coords)
    if n < 3:
        return np.nan, None, None

    d = pdist(coords)
    r_vals = np.logspace(np.log10(r_min), np.log10(r_max), n_bins)

    C = np.array([(d < r).sum() for r in r_vals], dtype=float)
    C /= (n * (n - 1) / 2.0)

    good = C > 0
    if good.sum() < 5:
        return np.nan, r_vals, C

    x = np.log10(r_vals[good])
    y = np.log10(C[good])

    coeff = np.polyfit(x, y, 1)
    D2 = coeff[0]
    return D2, r_vals, C

D2, D2_rvals, D2_C = correlation_dimension(coords_pc)

# ============================================================
# SAVE GLOBAL SUMMARY
# ============================================================
global_stats = pd.DataFrame({
    "statistic": [
        "N_regions",
        "pc_per_pixel",
        "inclination_deg",
        "position_angle_deg",
        "Clark_Evans_R",
        "mean_observed_nearest_neighbor_pc",
        "mean_CSR_nearest_neighbor_pc",
        "mean_MST_edge_pc",
        "correlation_dimension_D2"
    ],
    "value": [
        n_regions,
        pc_per_pixel,
        inclination_deg,
        pa_deg,
        R_ce,
        nn_obs_mean,
        nn_exp_mean,
        mst_mean_pc,
        D2
    ]
})

# Ripley / pair-correlation profiles
ripley_df = pd.DataFrame({
    "r_pc": radii_pc,
    "Ripley_K": K_r,
    "Besag_L_minus_r_pc": L_r
})

pcf_df = pd.DataFrame({
    "r_center_pc": r_centers,
    "pair_correlation_g_r": g_r
})

# ============================================================
# SAVE CATALOG + SUMMARY FILES
# ============================================================
catalog_output_path = os.path.join(
    CATALOG_DIR,
    "total_flux_catalog_with_deprojected_clustering_metrics.csv"
)
df.to_csv(catalog_output_path, index=False)

global_output_path = os.path.join(
    CATALOG_DIR,
    "clustering_global_statistics.csv"
)
global_stats.to_csv(global_output_path, index=False)

ripley_output_path = os.path.join(
    CATALOG_DIR,
    "clustering_ripley_profile.csv"
)
ripley_df.to_csv(ripley_output_path, index=False)

pcf_output_path = os.path.join(
    CATALOG_DIR,
    "clustering_pair_correlation_profile.csv"
)
pcf_df.to_csv(pcf_output_path, index=False)

print("Saved catalog:", catalog_output_path)
print("Saved global stats:", global_output_path)
print("Saved Ripley profile:", ripley_output_path)
print("Saved pair-correlation profile:", pcf_output_path)
print()
print("Clark-Evans R =", R_ce)
print("Mean MST edge [pc] =", mst_mean_pc)
print("Correlation dimension D2 =", D2)

pc/pixel = 1.3699
Saved catalog: CATALOGS/flux_catalogs/total_flux_catalog_with_deprojected_clustering_metrics.csv
Saved global stats: CATALOGS/flux_catalogs/clustering_global_statistics.csv
Saved Ripley profile: CATALOGS/flux_catalogs/clustering_ripley_profile.csv
Saved pair-correlation profile: CATALOGS/flux_catalogs/clustering_pair_correlation_profile.csv

Clark-Evans R = 0.8015251655839895
Mean MST edge [pc] = 28.65081803534059
Correlation dimension D2 = 1.9157910881515563
